# Multipath Single (Tx1/Rx1)

Single Tx/Rx notebook with optional fixed Tx/Rx.

In [ ]:
import numpy as np
from pathlib import Path
from sionna.rt import load_scene

from utils import (
    get_object_vertices,
    prepare_path_data,
    make_fixed_path_data,
    setup_tx_rx,
    solve_paths_for_frame,
    plot_path_from_indices,
)

np.set_printoptions(precision=3, suppress=True)


In [ ]:
# 1) Config
XML_PATH = Path('/data/hw/sionna/workspace/scenes/etri_0226/etri_260226_fixed.xml')
MERGE_SHAPES = False
RX_OBJECT_ID = 'car'   # used when RX_FIXED=False
TX_OBJECT_ID = 'car'   # used when TX_FIXED=False

RX_PATH_INDICES = [1, 14, 0, 12, 10, 8, 5, 4]
TX_PATH_INDICES = [20, 20]

RX_Z_OFFSET = 1.5
RX_SPEED_MS = 100.0 / 3.6
TX_Z_OFFSET = 20.0
TX_SPEED_MS = 0.0
DELTA_T = 0.2

RX_FIXED = False
RX_FIXED_POS = [80.0, -10.0, 1.5]
TX_FIXED = True
TX_FIXED_POS = [100.0, 20.0, 30.0]

TX_NAME = 'tx1'
RX_NAME = 'rx1'
TX_POWER_DBM = 43.0

SOLVER_KWARGS = dict(
    max_depth=3,
    samples_per_src=100000,
    diffuse_reflection=True,
    diffraction=True,
    synthetic_array=True,
)


In [ ]:
# 2) Load scene and vertices
scene = load_scene(str(XML_PATH), merge_shapes=MERGE_SHAPES)
rx_positions = None if RX_FIXED else get_object_vertices(scene, RX_OBJECT_ID, round_decimals=3)
tx_positions = None if TX_FIXED else get_object_vertices(scene, TX_OBJECT_ID, round_decimals=3)
print('scene loaded:', XML_PATH.resolve())
print('RX_FIXED =', RX_FIXED, '| TX_FIXED =', TX_FIXED)


In [ ]:
# 3) Build path_data
if RX_FIXED:
    rx_path_data = make_fixed_path_data(RX_FIXED_POS, delta_t=DELTA_T)
else:
    rx_path_indices = np.asarray(RX_PATH_INDICES, dtype=np.int32)
    rx_path_data = prepare_path_data(rx_positions, rx_path_indices, z_offset=RX_Z_OFFSET, speed_ms=RX_SPEED_MS, delta_t=DELTA_T)
    plot_path_from_indices(rx_positions, rx_path_indices, figsize=(10, 8))

if TX_FIXED:
    tx_path_data = make_fixed_path_data(TX_FIXED_POS, delta_t=DELTA_T)
else:
    tx_path_indices = np.asarray(TX_PATH_INDICES, dtype=np.int32)
    tx_path_data = prepare_path_data(tx_positions, tx_path_indices, z_offset=TX_Z_OFFSET, speed_ms=TX_SPEED_MS, delta_t=DELTA_T)
    plot_path_from_indices(tx_positions, tx_path_indices, figsize=(10, 8))

print('rx frames =', len(rx_path_data['frame_times']))
print('tx frames =', len(tx_path_data['frame_times']))


In [ ]:
# 4) Setup Tx/Rx
tx_start_pos = tx_path_data['waypoints'][0].tolist()
rx_start_pos = rx_path_data['waypoints'][0].tolist()
tx_names, rx_names, solver = setup_tx_rx(
    scene=scene,
    tx_positions=[tx_start_pos],
    rx_start_positions=[rx_start_pos],
    tx_names=[TX_NAME],
    rx_names=[RX_NAME],
    tx_power_dbm=TX_POWER_DBM,
)
print('tx_names =', tx_names, '| rx_names =', rx_names)


In [ ]:
# 5) Solve one frame
FRAME_IDX = 0
paths, rx_pos, rx_vel, tx_pos, tx_vel = solve_paths_for_frame(
    scene=scene,
    solver=solver,
    rx_names=rx_names,
    rx_path_data=rx_path_data,
    tx_names=tx_names,
    tx_path_data=tx_path_data,
    frame_idx=FRAME_IDX,
    tx_look_at_rx=True,
    tx_look_at_rx_idx=0,
    **SOLVER_KWARGS,
)
print('rx_pos=', rx_pos[0], 'tx_pos=', tx_pos[0])
scene.preview(paths=paths, show_devices=True, resolution=[900, 600])
